# ======================================================
# RETAIL AI SYSTEM — REAL ML MODELS (PRODUCTION UPGRADE)
# Demand Forecasting (LSTM + XGBoost) + Price Elasticity
# ======================================================


In [ ]:
import pandas as pd
import numpy as np

In [ ]:
def generate_dummy_data(n=1000):
    np.random.seed(42)
    data = pd.DataFrame({
        "sku": np.random.choice(["A", "B", "C"], n),
        "price": np.random.uniform(10, 100, n),
        "promo": np.random.randint(0, 2, n),
        "day": np.arange(n),
    })
    data["demand"] = 200 - 1.5 * data["price"] + 20 * data["promo"] + np.random.normal(0, 10, n)
    return data

print(generate_dummy_data())

In [ ]:
# ======================================================
# 2. XGBOOST DEMAND FORECAST MODEL
# ======================================================

from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split

class XGBoostDemandModel:
    def __init__(self):
        self.model = XGBRegressor(n_estimators=100, max_depth=5)

    def train(self, df):
        X = df[["price", "promo", "day"]]
        #print(X)
        y = df["demand"]
        #print(y)
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
        self.model.fit(X_train, y_train)
        print("XGBoost trained")

    def predict(self, price, promo, day):
        return float(self.model.predict([[price, promo, day]])[0])

a=XGBoostDemandModel()
print(a.train(generate_dummy_data()))
print(a.predict(50,1,120))

In [ ]:
# ======================================================
# 3. LSTM DEMAND FORECAST MODEL
# ======================================================

import torch
import torch.nn as nn

class LSTMModel(nn.Module):
    def __init__(self, input_size=1, hidden_size=50):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

class LSTMDemandModel:
    def __init__(self):
        self.model = LSTMModel()
        self.criterion = nn.MSELoss()
        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=0.01)

    def train(self, series, epochs=5):
        X, y = [], []
        seq_len = 10
        for i in range(len(series) - seq_len):
            X.append(series[i:i+seq_len])
            y.append(series[i+seq_len])
        print(X)
        print(y)
        X = torch.tensor(X).float().unsqueeze(-1)
        y = torch.tensor(y).float()

        for epoch in range(epochs):
            pred = self.model(X).squeeze()
            loss = self.criterion(pred, y)
            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()
        print("LSTM trained")

    def predict(self, seq):
        seq = torch.tensor(seq).float().unsqueeze(0).unsqueeze(-1)
        return float(self.model(seq).item())

sts = LSTMDemandModel()
# Extract the specific column containing the demand values as a list
#real_series = df["daily_demand"].tolist()
real_series =  generate_dummy_data()["demand"].tolist()
print(sts.train(real_series))
